# Groups and linked plots

To compare hosts, days or runs, put them in **one plot as groups** (`group=`): one time
axis, one toolbar, one legend. For plots that live apart, **link** them so they zoom
together. Docs: <https://mluttikh.github.io/anytimeline/how-to/groups.html>

In [ ]:
import numpy as np

from anytimeline import Timeline
from anytimeline.synthetic import service_log, service_metrics, traces

services = ["gateway", "auth", "api", "cache", "db"]
log = service_log(1200)
host = np.where(np.isin(log["user"], [f"u{i:02d}" for i in range(30)]), "web-1", "web-2")
log = {**log, "host": host}

## Two hosts in one plot

Each host gets a header with its own lanes, in the same order. Click a host's name to fold
it; a selection box can span both.

In [ ]:
hosts = Timeline(title="Two web servers", height=520, lane_order=services)
hosts.events(log, time="time", lane="service", group="host", color="level", symbol="event")
hosts

## Compared from their start

`align="start"` starts each group at 0: here the slowest and the fastest of twelve requests,
one above the other, on an axis that counts from their start.

In [ ]:
tr = traces(12, seed=3)
roots = tr.spans["parent_id"] == None  # noqa: E711
took = (tr.spans["end"] - tr.spans["start"])[roots]
slow, fast = tr.spans["trace_id"][roots][[took.argmax(), took.argmin()]]
pick = np.isin(tr.spans["trace_id"], [slow, fast])
pair = {k: v[pick] for k, v in tr.spans.items()}
pair["request"] = np.where(pair["trace_id"] == slow, "slowest", "fastest")

Timeline(title="Slowest and fastest, from their start", align="start", height=520).spans(
    pair, color="service", group="request")

## Linked plots

Two separate plots, linked: zoom or pan one and the other follows; the pointer's time shows
as a dashed line in both. They can be in different cells: run the next two cells, then zoom
in either plot.

In [ ]:
events_plot = Timeline(title="web-1: events", height=260, lane_order=services)
events_plot.events({k: v[log["host"] == "web-1"] for k, v in log.items()},
                   time="time", lane="service", color="level")
events_plot

In [ ]:
cpu_plot = Timeline(title="CPU", height=260, lane_order=services)
cpu_plot.values(service_metrics(), value="cpu", lane="service", kind="area")
cpu_plot.link(events_plot)
cpu_plot